# 第 11 堂：模型評估（混淆矩陣、Precision、Recall）
LESSON 11 · MODEL EVALUATION

### 本堂學習目標
學完後，你應該能夠：
1. 說出「只看正確率」為什麼會被騙。
2. 看懂**混淆矩陣**的四個格子：TP、FN、FP、TN。
3. 算出並解釋 **Precision（精確率）** 和 **Recall（召回率）**。
4. 依照商業情境，判斷應該重視 Precision 還是 Recall。
5. 用 `classification_report` 一次看完所有指標。
6. 調整判斷門檻，在 Precision 和 Recall 之間取捨。

### 本堂上課順序
| 順序 | 內容 |
| :--- | :--- |
| 0. 錯題複習 | 第 10 堂小考錯的三題 |
| 1. 生活例子 | 一個「什麼都不學」的模型，正確率 84%？ |
| 2. 概念解說 | 混淆矩陣、Precision、Recall |
| 3. 程式教學 | 基準模型 → KNN → 混淆矩陣 → 報告 → 調門檻 → 比較模型 |
| 4. 逐行解析 | 每段程式後面都有解析表 |
| 5. 實際練習 | 4 個練習，附參考解答 |
| 6. 小考驗收 | 自我檢查，再回對話做小考 |

---
## 零、錯題複習（第 10 堂第 7、11、18 題）

### 第 7 題：手算正確率 —— 一題一題對答案
```
y_test = ["續約", "流失", "續約", "流失", "續約"]
y_pred = ["續約", "續約", "續約", "流失", "流失"]
```
| 第幾題 | 實際 | 預測 | 對錯 |
| :--- | :--- | :--- | :--- |
| 1 | 續約 | 續約 | ○ |
| 2 | 流失 | 續約 | ✕ |
| 3 | 續約 | 續約 | ○ |
| 4 | 流失 | 流失 | ○ |
| 5 | 續約 | 流失 | ✕ |

答對 3 題 ÷ 5 題 = **0.6**。
**作法：直的對齊、一列一列比，不要整排一起看。** 今天的混淆矩陣，就是把這張表的 ○ 和 ✕ 再分得更細。

### 第 11 題：迴圈每一輪都會執行一次
```python
for k in [1, 3, 5]:
    print(k * 2)
```
| 第幾輪 | k | 執行 `print(k * 2)` | 印出 |
| :--- | :--- | :--- | :--- |
| 1 | 1 | print(2) | 2 |
| 2 | 3 | print(6) | 6 |
| 3 | 5 | print(10) | 10 |

`print` 在迴圈**裡面**（有縮排），所以每一輪都印一行，總共印 **3 行**，不會變成一個清單 `[2, 6, 10]`。
**作法：遇到迴圈，就像上面一樣畫一張「第幾輪」的表。** 今天步驟 6 也會用到迴圈。

### 第 18 題：多特徵的斜率，一定要加「其他特徵相同時」
> 營業額 = 3 × 廣告費 + 5 × 店員數 + 20

想像兩家店，**廣告費一樣**都是 10 萬：
- A 店 2 位店員：3 × 10 + 5 × 2 + 20 = 60
- B 店 3 位店員：3 × 10 + 5 × 3 + 20 = 65

只差 1 位店員，營業額差 **5**。但如果 B 店的廣告費也不一樣，差距就不只 5 了。
所以斜率 5 的正確白話是：「**在廣告費相同的情況下**，每多 1 位店員，營業額大約多 5。」

另外，「斜率 5 比斜率 3 大，所以店員比廣告費重要」是**錯的**。廣告費的單位是「萬元」、店員的單位是「人」，單位不同不能直接比大小（第 14 堂會教怎麼比）。

---
## 一、生活例子：一個「什麼都不學」的模型，正確率 84%？

延續第 10 堂的健身房。這次店長拿到 200 位會員的資料，但情況比較真實：
- 續約：169 位
- 流失：**只有 31 位**（大約 15%）

一位新來的工讀生說：「我做了一個超準的模型，正確率 84%！」
店長一看程式碼，發現工讀生的模型是這樣：

> **不管是誰，一律猜「續約」。**

因為大部分會員本來就會續約，所以「一律猜續約」也能答對 84%。
但店長真正想要的，是**找出會流失的人**，這個模型一個都沒找到！

> **當資料的類別很不平均時，正確率會騙人。**

這種情況在實務上非常常見：

| 情境 | 少數的那一類 | 大約比例 |
| :--- | :--- | :--- |
| 信用卡詐騙偵測 | 詐騙交易 | 不到 1% |
| 疾病篩檢 | 患病的人 | 可能只有幾 % |
| 工廠瑕疵檢測 | 瑕疵品 | 很少 |
| 會員流失 | 流失的會員 | 10～20% |

而且**少數的那一類，通常正是我們最在乎的**。所以需要更好的評估方法。

---
## 二、概念解說

### 1. 先決定「正類」（Positive）
「正類」不是「好的那一類」，而是**我們想要找出來的那一類**。
今天要找的是會流失的會員，所以：**流失 = 正類（Positive）**，續約 = 負類（Negative）。

### 2. 混淆矩陣（Confusion Matrix）
把第 10 堂的「○ 和 ✕」再分成四種：

| | **預測：流失** | **預測：續約** |
| :--- | :--- | :--- |
| **實際：流失** | **TP** 真正例：抓到了 ✓ | **FN** 偽負例：**漏掉了** ✗ |
| **實際：續約** | **FP** 偽正例：**誤報了** ✗ | **TN** 真負例：正確放過 ✓ |

記法（第一個字母看「對不對」，第二個字母看「模型猜什麼」）：
- **T**rue / **F**alse：模型猜得對不對
- **P**ositive / **N**egative：模型猜的是正類（流失）還是負類（續約）

| 代號 | 白話 | 健身房的後果 |
| :--- | :--- | :--- |
| TP | 預測流失，真的流失 | 教練打電話，成功挽留的機會 ✓ |
| FN | 預測續約，結果流失了 | **沒人關心他，就這樣流失了** |
| FP | 預測流失，結果他本來就會續約 | **教練白打一通電話** |
| TN | 預測續約，真的續約 | 不用處理 ✓ |

正確率 = (TP + TN) ÷ 全部。**它把 FN 和 FP 混在一起算，看不出模型錯在哪裡。**

### 3. Precision（精確率）和 Recall（召回率）

| 指標 | 公式 | 回答的問題 | 健身房白話 |
| :--- | :--- | :--- | :--- |
| **Precision** 精確率 | TP ÷ (TP + FP) | 模型說「會流失」的人裡，**有幾成真的會流失**？ | 教練打的電話裡，有幾通是**打對人**的？ |
| **Recall** 召回率 | TP ÷ (TP + FN) | 真正會流失的人裡，**被模型抓到幾成**？ | 會流失的會員，有幾成**有被關心到**？ |

記憶方式：
- **Precision 看「預測那一欄」**：我說是的，準不準？（**寧缺勿濫**）
- **Recall 看「實際那一列」**：真的是的，抓到幾個？（**寧可錯殺，不可放過**）

### 4. 該重視哪一個？看「哪種錯誤比較嚴重」

| 情境 | 比較怕哪種錯？ | 重視 |
| :--- | :--- | :--- |
| 疾病篩檢 | 漏掉病人（FN），會延誤治療 | **Recall** |
| 詐騙偵測 | 漏掉詐騙（FN），損失很大 | **Recall** |
| 垃圾郵件過濾 | 把重要信件丟進垃圾桶（FP） | **Precision** |
| 健身房挽留 | 漏掉流失的人（FN）比白打一通電話（FP）嚴重 | 偏重 **Recall** |

兩個指標通常**此消彼長**：想抓到更多（Recall ↑），就會多一些誤報（Precision ↓）。等一下會親眼看到。

### 5. F1 分數
如果兩個都重要，可以看 **F1 分數**：Precision 和 Recall 的綜合分數，兩個都高，F1 才會高。現在只要知道它的意思就好。

---
## 三、程式教學

### 步驟 0：匯入工具與中文字型設定

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             accuracy_score, precision_score, recall_score,
                             classification_report)

# 中文字型設定（同前幾堂）
import sys, os
import matplotlib.font_manager as fm
if "google.colab" in sys.modules:
    os.system("apt-get -qq install -y fonts-noto-cjk > /dev/null")
noto = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
if os.path.exists(noto):
    fm.fontManager.addfont(noto)
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "Microsoft JhengHei", "PingFang TC", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

print("準備完成")

準備完成


#### 新的匯入
`sklearn.metrics` 是 sklearn 的「評估指標」工具箱（metrics = 衡量標準）。

| 名稱 | 用途 |
| :--- | :--- |
| `confusion_matrix` | 算出混淆矩陣 |
| `ConfusionMatrixDisplay` | 把混淆矩陣畫成圖 |
| `accuracy_score`、`precision_score`、`recall_score` | 正確率、精確率、召回率 |
| `classification_report` | 一次印出所有指標 |

匯入很多東西時，可以用小括號 `( )` 包起來換行，比較好讀。

### 步驟 1：準備資料

200 位會員，3 個特徵：每月到店次數、會員年資、**客訴次數**（新的）。
這格資料很長，**直接執行即可，不用看內容**。

In [ ]:
gym = pd.DataFrame({
    "到店次數": [11, 10, 10, 10, 8, 12, 1, 9, 10, 8, 11, 12, 1, 2, 12, 7, 4, 5, 11, 4,
            7, 4, 11, 2, 9, 12, 7, 3, 0, 9, 7, 6, 10, 8, 10, 9, 8, 0, 7, 0,
            8, 12, 11, 3, 12, 5, 6, 3, 8, 3, 12, 4, 0, 1, 6, 4, 9, 7, 5, 12,
            12, 2, 1, 12, 1, 10, 3, 5, 2, 11, 0, 11, 2, 12, 6, 2, 0, 9, 10, 10,
            11, 8, 8, 11, 2, 10, 10, 2, 2, 8, 6, 2, 8, 2, 6, 6, 5, 4, 3, 12,
            4, 0, 5, 2, 11, 10, 2, 11, 1, 0, 7, 9, 6, 9, 9, 8, 6, 7, 3, 11,
            10, 11, 3, 5, 8, 0, 6, 12, 8, 3, 7, 2, 8, 0, 6, 6, 4, 12, 6, 9,
            11, 5, 2, 10, 6, 8, 1, 11, 11, 11, 8, 5, 1, 10, 11, 9, 7, 11, 9, 3,
            10, 3, 9, 6, 10, 2, 6, 1, 12, 6, 4, 4, 11, 5, 4, 4, 12, 3, 12, 11,
            7, 9, 4, 5, 1, 5, 8, 3, 8, 5, 8, 7, 11, 0, 11, 2, 2, 3, 7, 11],
    "會員年資": [3.1, 6.3, 0.5, 4.2, 5.8, 0.1, 4.3, 0.5, 2.6, 3.8, 6.9, 0.6, 0.5, 5.1, 2.2, 5.1, 5.1, 1.6, 0.2, 3.4,
            2.4, 5.4, 6.4, 2.7, 1.7, 6.5, 2.5, 0.3, 4.3, 5.0, 7.5, 3.5, 2.8, 1.7, 4.1, 2.7, 5.4, 4.5, 6.1, 7.2,
            3.8, 0.4, 3.3, 3.6, 2.6, 0.4, 6.1, 7.8, 3.7, 2.6, 3.6, 3.5, 0.0, 7.2, 7.6, 4.5, 1.1, 3.3, 4.1, 3.0,
            1.9, 6.2, 5.0, 3.6, 1.7, 5.1, 7.7, 1.8, 1.3, 0.3, 6.8, 6.1, 5.1, 2.4, 6.2, 0.2, 5.4, 0.8, 3.9, 6.3,
            2.3, 7.8, 6.8, 2.6, 0.4, 2.9, 3.4, 7.6, 7.5, 6.1, 1.9, 7.1, 8.0, 3.7, 5.0, 5.0, 3.6, 3.5, 6.2, 1.6,
            0.3, 2.7, 5.7, 6.8, 0.4, 6.9, 5.3, 6.2, 2.4, 6.0, 0.1, 7.5, 2.7, 6.1, 5.3, 6.5, 7.6, 6.2, 1.9, 2.1,
            1.8, 7.0, 6.4, 7.0, 5.3, 4.4, 0.2, 6.3, 8.0, 3.4, 1.1, 7.0, 1.9, 4.1, 3.0, 5.2, 2.9, 6.5, 3.6, 5.2,
            4.0, 7.1, 2.5, 6.4, 2.5, 6.0, 4.7, 1.6, 5.7, 2.0, 4.6, 0.5, 0.5, 2.4, 4.2, 7.0, 4.8, 5.2, 6.1, 3.3,
            6.9, 0.4, 4.4, 4.7, 7.7, 4.9, 6.9, 7.2, 3.0, 5.5, 3.4, 3.1, 2.6, 4.3, 5.7, 0.6, 2.7, 0.8, 2.4, 7.9,
            3.9, 0.4, 1.7, 3.6, 4.3, 3.8, 3.7, 3.6, 7.2, 1.7, 3.1, 2.0, 1.3, 4.9, 6.3, 4.6, 1.3, 5.3, 7.5, 1.7],
    "客訴次數": [0, 1, 0, 0, 1, 0, 2, 0, 0, 0, 2, 1, 0, 0, 0, 0, 0, 0, 1, 2,
            1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0,
            0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1,
            2, 1, 0, 1, 1, 0, 1, 1, 0, 2, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0,
            1, 0, 0, 1, 0, 1, 0, 3, 1, 1, 0, 1, 2, 0, 1, 1, 1, 0, 0, 1,
            2, 0, 1, 0, 1, 2, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0,
            0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 2, 2, 1, 0, 1, 0, 0, 0, 0,
            0, 0, 0, 0, 2, 0, 2, 0, 1, 0, 1, 2, 1, 1, 3, 0, 1, 0, 0, 0,
            3, 0, 0, 0, 0, 1, 0, 1, 3, 1, 0, 0, 0, 1, 0, 3, 2, 2, 1, 0,
            0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 2, 0],
    "流失標記": [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
            0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
            0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
            0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0,
            0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
            1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
            0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0,
            0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
            0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0,
            0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0]
})

# 把 1／0 轉成文字類別（第 10 堂的 np.where）
gym["結果"] = np.where(gym["流失標記"] == 1, "流失", "續約")

print(gym.head())
print("-" * 30)
print(gym["結果"].value_counts())

#### 逐行解析

| 程式碼 | 意思 |
| :--- | :--- |
| `"流失標記": [0, 0, 1, ...]` | 原始資料用 1 代表流失、0 代表續約 |
| `np.where(條件, "流失", "續約")` | 第 10 堂：條件成立填「流失」，否則填「續約」 |
| `value_counts()` | 第 10 堂：每一類各幾個 |

#### 觀察
續約 169、流失 31，流失只占 15.5%。這叫做**類別不平衡**。

### 步驟 2：切分資料

In [ ]:
X = gym[["到店次數", "會員年資", "客訴次數"]]
y = gym["結果"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("測試集：", X_test.shape)
print(y_test.value_counts())

#### 逐行解析
和第 10 堂一樣。這裡 `stratify=y` **特別重要**：流失的人很少，如果不分層抽樣，測試集可能只分到 2、3 個流失的人，甚至 0 個，就沒辦法評估了。

測試集 50 位：續約 42、流失 **8**。等一下所有指標，都是在看模型能不能找出這 8 個人。

### 步驟 3：工讀生的模型 —— 一律猜「續約」

In [ ]:
y_lazy = np.array(["續約"] * len(y_test))

print("一律猜續約的正確率：", accuracy_score(y_test, y_lazy))
print("找到幾個流失的人：", (y_lazy == "流失").sum())

#### 逐行解析

| 程式碼 | 意思 |
| :--- | :--- |
| `["續約"] * len(y_test)` | 把 `"續約"` 重複 50 次，做成一個清單（`len()` 是長度，第 6 堂用過） |
| `accuracy_score(y_test, y_lazy)` | 正確率，等於第 10 堂的 `(y_lazy == y_test).mean()` |

#### 結果
正確率 **0.84**，看起來還不錯，但**一個流失的人都沒找到**。
這種「什麼都沒學、一律猜最多的那一類」的模型，叫做**基準模型**（Baseline）。

> **你的模型至少要比基準模型好，才有意義。** 以後看到正確率，先想想「一律猜多數」能拿幾分。

### 步驟 4：訓練 KNN，看混淆矩陣

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

print("KNN 正確率：", accuracy_score(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=["流失", "續約"])
print(cm)

#### 逐行解析

| 程式碼 | 意思 |
| :--- | :--- |
| `confusion_matrix(實際, 預測, labels=[...])` | 算出混淆矩陣。**順序：先實際、再預測** |
| `labels=["流失", "續約"]` | 指定類別順序：把正類「流失」放第一個，矩陣才會是 TP 在左上 |

印出來的是一個 2×2 的陣列，但沒有標題不好讀。我們把它變成 DataFrame：

In [ ]:
cm_table = pd.DataFrame(
    cm,
    index=["實際：流失", "實際：續約"],
    columns=["預測：流失", "預測：續約"]
)
print(cm_table)

#### 逐行解析

| 程式碼 | 意思 |
| :--- | :--- |
| `pd.DataFrame(cm, index=..., columns=...)` | 把陣列變成有名字的表格。`index` 是列名、`columns` 是欄名 |

#### 讀混淆矩陣

| | 預測：流失 | 預測：續約 | 小計 |
| :--- | :--- | :--- | :--- |
| **實際：流失** | **TP = 5** 抓到了 | **FN = 3** 漏掉了 | 8 位真的流失 |
| **實際：續約** | **FP = 2** 誤報 | **TN = 40** 正確放過 | 42 位真的續約 |
| **小計** | 7 位被說會流失 | 43 位被說會續約 | 共 50 位 |

- 正確率 = (5 + 40) ÷ 50 = **0.90**（只比基準模型高 0.06）
- 8 位會流失的人，抓到 **5** 位、漏掉 **3** 位
- 模型說會流失的 7 位裡，有 **2** 位其實會續約（教練白打電話）

這些資訊，光看「正確率 0.90」是完全看不出來的。

In [ ]:
# 畫成圖
ConfusionMatrixDisplay(cm, display_labels=["流失", "續約"]).plot(cmap="Blues")
plt.title("KNN 的混淆矩陣")
plt.xlabel("預測")
plt.ylabel("實際")
plt.show()

#### 逐行解析

| 程式碼 | 意思 |
| :--- | :--- |
| `ConfusionMatrixDisplay(cm, display_labels=...)` | 建立混淆矩陣的圖 |
| `.plot(cmap="Blues")` | 畫出來，用藍色系，數字越大顏色越深 |

圖的**左上到右下**這條對角線（TP、TN）是**答對**的；另外兩格（FN、FP）是**答錯**的。

注意：左上的 TP 顏色很淺，不是因為模型不好，而是流失的人本來就只有 8 位，數字小、顏色就淺。**類別不平衡時，要看格子裡的數字，不要只看顏色。**

### 步驟 5：算出 Precision 和 Recall

In [ ]:
# 自己用公式算
TP, FN, FP, TN = 5, 3, 2, 40

print("Precision = TP ÷ (TP + FP) =", TP, "÷", TP + FP, "=", round(TP / (TP + FP), 3))
print("Recall    = TP ÷ (TP + FN) =", TP, "÷", TP + FN, "=", round(TP / (TP + FN), 3))

# 用 sklearn 算
print("-" * 30)
print("precision_score：", round(precision_score(y_test, y_pred, pos_label="流失"), 3))
print("recall_score   ：", round(recall_score(y_test, y_pred, pos_label="流失"), 3))

#### 逐行解析

| 程式碼 | 意思 |
| :--- | :--- |
| `TP, FN, FP, TN = 5, 3, 2, 40` | 一次設定四個變數（和 `train_test_split` 一次接四個結果一樣） |
| `/` | 除法 |
| `pos_label="流失"` | **告訴 sklearn 正類是哪一個**。類別是文字時一定要寫，不然它不知道要算誰 |

#### 白話解讀
- **Precision 0.714**：模型說「會流失」的 7 位裡，有 5 位真的會流失 → 教練打的電話，**約 7 成打對人**。
- **Recall 0.625**：真正會流失的 8 位裡，模型抓到 5 位 → **約 6 成的流失會員有被關心到**，還有 3 位漏掉了。

對店長來說，漏掉 3 位比較可惜。能不能讓 Recall 高一點？請看步驟 7。

### 步驟 6：classification_report —— 一次看全部

In [ ]:
print(classification_report(y_test, y_pred))

#### 怎麼讀這張報告

| 欄位 | 意思 |
| :--- | :--- |
| `precision` | 精確率 |
| `recall` | 召回率 |
| `f1-score` | F1 分數（綜合 precision 和 recall） |
| `support` | 這一類在測試集裡**實際有幾個**（流失 8、續約 42） |

| 列 | 意思 |
| :--- | :--- |
| `流失` 那一列 | 把「流失」當正類算出來的指標，**我們最關心這一列** |
| `續約` 那一列 | 把「續約」當正類算的指標，數字通常都很高，因為續約的人多、很好猜 |
| `accuracy` | 正確率 |
| `macro avg`、`weighted avg` | 兩類的平均，初學可以先不看 |

> **重點：** 類別不平衡時，要看**少數那一類（流失）**的 precision 和 recall，不要被續約那一列的高分騙了。

### 步驟 7：調整門檻 —— 用 Precision 換 Recall

第 10 堂學過，KNN 是 5 個鄰居投票，`predict_proba` 就是投票比例。
`predict()` 預設的規則是：**流失票數超過一半（3 票以上）才判斷為流失**。

如果店長說：「寧可多打幾通電話，也不要漏掉會流失的人」，我們可以**降低門檻**：只要有 2 票、甚至 1 票說會流失，就當作會流失。

In [ ]:
print("類別順序：", knn.classes_)
churn_proba = knn.predict_proba(X_test)[:, 0]   # 第 0 欄是「流失」的機率
print("前 10 位的流失機率：", churn_proba[:10])

#### 逐行解析

| 程式碼 | 意思 |
| :--- | :--- |
| `knn.classes_` | 類別順序是 `['流失', '續約']`，所以流失是第 0 欄 |
| `predict_proba(X_test)[:, 0]` | `[:, 0]` = 「每一列都取第 0 欄」，也就是每位會員的流失機率（第 10 堂練習三用過 `[:, 1]`） |
| `churn_proba[:10]` | 只看前 10 個 |

機率只會是 0、0.2、0.4、0.6、0.8、1.0，因為 5 個鄰居投票，每票是 0.2。

In [ ]:
for threshold in [0.5, 0.3, 0.1]:
    y_new = np.where(churn_proba >= threshold, "流失", "續約")
    p = precision_score(y_test, y_new, pos_label="流失")
    r = recall_score(y_test, y_new, pos_label="流失")
    print("門檻", threshold,
          "  Precision：", round(p, 3),
          "  Recall：", round(r, 3),
          "  預測流失人數：", (y_new == "流失").sum())

#### 逐行解析（照錯題複習的方法，畫出「第幾輪」的表）

| 第幾輪 | threshold | 意思 | 印出一行結果 |
| :--- | :--- | :--- | :--- |
| 1 | 0.5 | 流失機率 ≥ 0.5，也就是 **3 票以上** → 判斷流失（和預設一樣） | 第 1 行 |
| 2 | 0.3 | **2 票以上** → 判斷流失 | 第 2 行 |
| 3 | 0.1 | **1 票以上** → 判斷流失 | 第 3 行 |

| 程式碼 | 意思 |
| :--- | :--- |
| `np.where(churn_proba >= threshold, "流失", "續約")` | 第 10 堂：機率超過門檻就填「流失」 |

#### 觀察：此消彼長

| 門檻 | 教練要打幾通電話 | Precision（打對人的比例） | Recall（流失會員被關心到的比例） |
| :--- | :--- | :--- | :--- |
| 0.5（預設） | 7 通 | 0.714 | 0.625 |
| 0.3 | 10 通 | 0.6 | 0.75 |
| 0.1 | 14 通 | 0.5 | **0.875** |

門檻越低 → 打越多電話 → 抓到越多流失的人（Recall ↑），但也越多電話是白打的（Precision ↓）。

**沒有標準答案，要看商業考量：**
- 如果打一通電話成本很低、流失一位會員損失很大 → 選低門檻（0.1），8 位抓到 7 位。
- 如果教練人力很少，只能打少數幾通 → 選預設門檻，打的電話比較準。

> **模型的工作是給出機率，門檻是商業決策。** 這就是資料分析師要跟主管討論的地方。

### 步驟 8：比較兩個模型 —— 正確率差不多，Recall 差很多

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)
y_tree = tree.predict(X_test)

summary = pd.DataFrame({
    "模型": ["一律猜續約", "KNN (K=5)", "決策樹 (深度 3)"],
    "正確率": [accuracy_score(y_test, y_lazy),
              accuracy_score(y_test, y_pred),
              accuracy_score(y_test, y_tree)],
    "Recall（抓到流失）": [0,
                      recall_score(y_test, y_pred, pos_label="流失"),
                      recall_score(y_test, y_tree, pos_label="流失")]
})
print(summary.round(3))

#### 逐行解析
建立一個比較表，每一欄是一個 Python 清單（第 5 堂的 DataFrame）。基準模型的 Recall 直接寫 0，因為它一個都沒抓到。

`summary.round(3)` 把整張表四捨五入到小數 3 位。

#### 觀察

| 模型 | 正確率 | Recall |
| :--- | :--- | :--- |
| 一律猜續約 | 0.84 | 0 |
| KNN | 0.90 | 0.625 |
| 決策樹 | 0.86 | **0.25** |

決策樹的正確率 0.86，只比 KNN 低一點點；但 8 位流失會員**只抓到 2 位**。
**如果只看正確率，你會以為兩個模型差不多；看了 Recall，才知道決策樹幾乎沒用。**

這就是本堂最重要的結論：

> **類別不平衡時，一定要看混淆矩陣、Precision 和 Recall，不能只看正確率。**

---
## 四、本堂重點整理

### 混淆矩陣（正類 = 我們想找的那一類）
| | 預測：正類 | 預測：負類 |
| :--- | :--- | :--- |
| **實際：正類** | TP 抓到了 | FN 漏掉了 |
| **實際：負類** | FP 誤報 | TN 正確放過 |

### 指標
| 指標 | 公式 | 白話 | 什麼時候重視 |
| :--- | :--- | :--- | :--- |
| Accuracy 正確率 | (TP+TN) ÷ 全部 | 全部答對幾成 | 類別平衡時 |
| Precision 精確率 | TP ÷ (TP+FP) | 說「是」的裡面，幾成真的是 | 誤報代價高（垃圾郵件） |
| Recall 召回率 | TP ÷ (TP+FN) | 真的「是」的裡面，抓到幾成 | 漏掉代價高（疾病、詐騙） |
| F1 | 兩者綜合 | 兩個都重要時看 | |

### 程式碼
| 程式碼 | 意思 |
| :--- | :--- |
| `confusion_matrix(y_test, y_pred, labels=[正類, 負類])` | 混淆矩陣，先實際、再預測 |
| `ConfusionMatrixDisplay(cm, display_labels=[...]).plot()` | 畫混淆矩陣 |
| `precision_score(y_test, y_pred, pos_label="流失")` | 精確率，文字類別要指定正類 |
| `recall_score(y_test, y_pred, pos_label="流失")` | 召回率 |
| `classification_report(y_test, y_pred)` | 一次看全部指標 |
| `np.where(proba >= 門檻, 正類, 負類)` | 自己調整判斷門檻 |

### 三個觀念
1. **先看基準模型**：「一律猜多數」能拿幾分？你的模型要比它好。
2. **此消彼長**：門檻調低 → Recall ↑、Precision ↓。
3. **門檻是商業決策**：看「漏掉」和「誤報」哪個代價比較高。

### 常見錯誤
| 狀況 | 原因 | 解法 |
| :--- | :--- | :--- |
| `pos_label=1 is not a valid label` | 類別是文字，沒指定正類 | 加上 `pos_label="流失"` |
| 混淆矩陣 TP 跑到右下角 | 沒指定 `labels` 順序 | 加上 `labels=["流失", "續約"]` |
| `confusion_matrix` 的實際、預測放反 | 參數順序錯 | **先實際（y_test）、再預測（y_pred）** |
| 測試集裡少數類只有一兩個 | 沒加 `stratify=y` | 加上 `stratify=y` |

---
## 五、實際練習

### 練習一：手算指標
某個詐騙偵測模型的混淆矩陣如下（正類 = 詐騙）：

| | 預測：詐騙 | 預測：正常 |
| :--- | :--- | :--- |
| 實際：詐騙 | 30 | 20 |
| 實際：正常 | 10 | 940 |

請用程式算出正確率、Precision、Recall，並用一句白話說明 Recall 的意思。

In [ ]:
# 練習一：在這裡寫你的程式

In [ ]:
# 練習一 參考解答
TP, FN, FP, TN = 30, 20, 10, 940
print("正確率：", (TP + TN) / (TP + FN + FP + TN))
print("Precision：", TP / (TP + FP))
print("Recall：", TP / (TP + FN))
# 正確率 0.97 看起來很高，但 Recall 只有 0.6：50 筆詐騙裡只抓到 30 筆，有 20 筆詐騙被放過了。
# 基準模型「一律猜正常」的正確率是 950 ÷ 1000 = 0.95，這個模型只比它好一點點。

### 練習二：判斷該重視哪個指標
用 `print` 寫下你的答案和理由：
1. 機場安檢偵測危險物品
2. 推薦系統挑出「使用者一定會喜歡」的 3 部電影放在首頁
3. 銀行用模型決定要不要**自動拒絕**一筆貸款申請

In [ ]:
# 練習二：在這裡寫你的答案

In [ ]:
# 練習二 參考解答
print("1. Recall：漏掉一個危險物品（FN）後果非常嚴重，寧可多檢查幾個包包（FP）。")
print("2. Precision：首頁只有 3 個位置，推薦的一定要準；漏掉一些使用者可能喜歡的電影沒關係。")
print("3. Precision：自動拒絕如果誤判（FP），會把好客戶趕走；不確定的可以交給人工審核。")

### 練習三：KNN 換成 K = 3
把 KNN 改成 `n_neighbors=3`，印出混淆矩陣（用 DataFrame 加上名字）和 `classification_report`。
流失那一列的 Recall，比 K = 5 高還是低？

In [ ]:
# 練習三：在這裡寫你的程式

In [ ]:
# 練習三 參考解答
knn3 = KNeighborsClassifier(n_neighbors=3)
knn3.fit(X_train, y_train)
y3 = knn3.predict(X_test)

cm3 = confusion_matrix(y_test, y3, labels=["流失", "續約"])
print(pd.DataFrame(cm3, index=["實際：流失", "實際：續約"], columns=["預測：流失", "預測：續約"]))
print(classification_report(y_test, y3))

### 練習四（挑戰題）：找出最適合店長的門檻
店長說：「我希望**至少抓到 7 成的流失會員**（Recall ≥ 0.7），在這個前提下，電話越少越好。」
請用步驟 7 的迴圈，試試門檻 0.5、0.3、0.1，找出符合條件、而且預測流失人數最少的門檻。

In [ ]:
# 練習四：在這裡寫你的程式

In [ ]:
# 練習四 參考解答
for threshold in [0.5, 0.3, 0.1]:
    y_new = np.where(churn_proba >= threshold, "流失", "續約")
    r = recall_score(y_test, y_new, pos_label="流失")
    n_calls = (y_new == "流失").sum()
    print("門檻", threshold, " Recall：", round(r, 3), " 要打幾通電話：", n_calls)
# 門檻 0.5 的 Recall 只有 0.625，不符合。
# 門檻 0.3 的 Recall = 0.75 ≥ 0.7，只要打 10 通；門檻 0.1 雖然 Recall 更高，但要打 14 通。
# → 選門檻 0.3（2 票以上就當作會流失）。

---
## 六、小考驗收

先不看筆記回答下面的問題，能回答出來再回到對話中做第 11 堂小考。

1. 為什麼「一律猜續約」的模型正確率有 84%？它有什麼問題？
2. 混淆矩陣的 FN 和 FP，在健身房的例子裡各代表什麼？
3. Precision 和 Recall 的公式各是什麼？
4. 疾病篩檢應該重視 Precision 還是 Recall？為什麼？
5. `confusion_matrix(y_test, y_pred)` 的兩個參數，哪個是實際、哪個是預測？
6. 類別是文字時，`precision_score` 要加什麼參數？
7. 把判斷門檻從 0.5 調低到 0.1，Precision 和 Recall 會怎麼變？
8. 兩個模型正確率是 0.90 和 0.86，Recall 是 0.625 和 0.25，類別不平衡時你會選哪一個？

<details>
<summary>點我看參考答案</summary>

1. 因為 84% 的會員本來就會續約。它一個流失的人都沒找到（Recall = 0），完全沒用。
2. FN：會流失卻被預測成續約，沒人關心就流失了。FP：本來會續約卻被預測成流失，教練白打一通電話。
3. Precision = TP ÷ (TP + FP)；Recall = TP ÷ (TP + FN)。
4. Recall。漏掉病人（FN）會延誤治療，比多做一次檢查（FP）嚴重。
5. 第一個是實際（y_test），第二個是預測（y_pred）。
6. `pos_label="流失"`（指定正類）。
7. Recall 上升（抓到更多），Precision 下降（誤報變多）。
8. 選正確率 0.90、Recall 0.625 的。Recall 0.25 代表大部分流失的人都沒抓到。

</details>

### 下一堂預告：第 12 堂 過度擬合與欠擬合
第 8、10 堂已經好幾次看到「訓練集滿分、測試集變差」的背答案現象。
第 12 堂要正式介紹：**過度擬合（Overfitting）**、**欠擬合（Underfitting）**，以及怎麼畫出**學習曲線**，找到模型「剛剛好」的複雜度。